# Notas — Aula 3: Recursão, exceções e arquivos — o robô autônomo

Marco do robô: **v4 → v5 → v6**. O robô aprende a explorar a grade sozinho (recursão) e a
ler seu programa de um arquivo de texto, gravando um log de onde passou. Depois desta aula
a D1 fecha com Python idiomático (Aula 4) — sem versão nova do robô, só um jeito melhor de
escrever o que já existe.

> ⚠️ **Antes de começar:** rode todas as células a partir do topo (**Run All**, ou `Kernel → Restart & Run All`) antes de pular para qualquer seção — a célula seguinte define constantes usadas no notebook inteiro. Se aparecer `NameError`, é sinal de que uma célula anterior não foi executada nesta sessão do kernel.


In [1]:
# Recap da Aula 2 — necessário para os exemplos de hoje (robô v4) — EXECUTE esta célula primeiro
LADO_GRADE = 5
DELTAS    = {"LESTE": (1, 0), "NORTE": (0, 1), "OESTE": (-1, 0), "SUL": (0, -1)}
GIRAR_ESQ = {"LESTE": "NORTE", "NORTE": "OESTE", "OESTE": "SUL", "SUL": "LESTE"}
GIRAR_DIR = {"LESTE": "SUL", "SUL": "OESTE", "OESTE": "NORTE", "NORTE": "LESTE"}


def posicao_valida(x, y):
    return 0 <= x < LADO_GRADE and 0 <= y < LADO_GRADE


def criar_robo(x=0, y=0, direcao="LESTE"):
    return {"x": x, "y": y, "direcao": direcao, "trajetoria": [(x, y)]}


def sensor_frente(robo, obstaculos):
    dx, dy = DELTAS[robo["direcao"]]
    nx, ny = robo["x"] + dx, robo["y"] + dy
    return posicao_valida(nx, ny) and (nx, ny) not in obstaculos


def avancar(robo, obstaculos):
    if sensor_frente(robo, obstaculos):
        dx, dy = DELTAS[robo["direcao"]]
        robo["x"] += dx
        robo["y"] += dy
        robo["trajetoria"].append((robo["x"], robo["y"]))
        return True
    return False


def girar(robo, lado):
    if lado == "ESQ":
        robo["direcao"] = GIRAR_ESQ[robo["direcao"]]
    elif lado == "DIR":
        robo["direcao"] = GIRAR_DIR[robo["direcao"]]


## 1. Recursão: caso-base (CB) e caso-recursivo (CR)

Uma função recursiva chama a si mesma com um problema **menor**. Toda função recursiva
precisa de dois ingredientes: o **caso-base** (a condição que não se chama de novo — devolve
a resposta direto) e o **caso-recursivo** (reduz o problema e chama a função de novo). Sem
caso-base, a recursão nunca para e o Python levanta `RecursionError`.

No robô, vamos usar exatamente esse padrão para andar pela grade sozinho (seção 3).


In [2]:
def fatorial(n):
    if n <= 1:                      # caso-base
        return 1
    return n * fatorial(n - 1)      # caso-recursivo


print(fatorial(5))    # 120
print(fatorial(0))    # 1


def soma_lista(L):
    if len(L) == 0:                     # caso-base: lista vazia
        return 0
    return L[0] + soma_lista(L[1:])     # caso-recursivo: primeiro + resto


print(soma_lista([3, 1, 4, 1, 5]))   # 14


120
1
14


### Sua vez

In [3]:
# soma_ate(n), recursiva, soma os inteiros de 1 até n.
def soma_ate(n):
    if n == 0:                  # caso-base
        return 0
    return n + soma_ate(n - 1)  # caso-recursivo


print(soma_ate(5))    # 15 (1+2+3+4+5)
print(soma_ate(0))    # 0


15
0


## 2. Visualizando a pilha de chamadas

Cada chamada recursiva fica **suspensa**, esperando o resultado da chamada de baixo — como
uma pilha de pratos. Adicionar `print`s de diagnóstico (um antes de recursar, outro depois)
deixa esse comportamento visível: primeiro todas as chamadas "empilham" (mensagens `->`),
depois os resultados "descem" na ordem inversa (mensagens `<-`).


In [4]:
def fatorial_diagnostico(n):
    print(f"  -> prestes a calcular fatorial({n})")
    if n <= 1:
        print("  <- caso-base: retorna 1")
        return 1
    resultado = n * fatorial_diagnostico(n - 1)
    print(f"  <- fatorial({n}) = {resultado}")
    return resultado


print(fatorial_diagnostico(4))


  -> prestes a calcular fatorial(4)
  -> prestes a calcular fatorial(3)
  -> prestes a calcular fatorial(2)
  -> prestes a calcular fatorial(1)
  <- caso-base: retorna 1
  <- fatorial(2) = 2
  <- fatorial(3) = 6
  <- fatorial(4) = 24
24


### Sua vez

In [5]:
# contagem_regressiva com prints de diagnóstico.
def contagem_regressiva(n):
    print(f"  -> contando {n}")
    if n == 0:
        print("  <- disparou!")
        return "disparou!"
    return contagem_regressiva(n - 1)


print(contagem_regressiva(3))


  -> contando 3
  -> contando 2
  -> contando 1
  -> contando 0
  <- disparou!
disparou!


## 3. Flood fill: o robô explora a grade (v5)

"Quantas células o robô alcança sem bater em obstáculo, a partir de onde está?" é
recursão *natural*: explorar `(x, y)` é explorar os quatro vizinhos, cada um do mesmo jeito.
Três casos-base (saiu da grade / é obstáculo / já foi contada) e um caso-recursivo (conta a
célula + soma os quatro vizinhos). Marcar a célula como visitada **antes** de chamar os
vizinhos é o que impede um loop infinito entre duas células vizinhas.


In [6]:
def flood_fill(robo, visitadas, x, y):
    if x < 0 or x >= LADO_GRADE or y < 0 or y >= LADO_GRADE:
        return 0                    # CB 1: saiu da grade
    if (x, y) in robo['obstaculos']:
        return 0                    # CB 2: obstáculo
    if (x, y) in visitadas:
        return 0                    # CB 3: já contamos esta célula
    visitadas.add((x, y))           # marcar ANTES de chamar os vizinhos
    conta = 1
    conta += flood_fill(robo, visitadas, x + 1, y)
    conta += flood_fill(robo, visitadas, x - 1, y)
    conta += flood_fill(robo, visitadas, x,     y + 1)
    conta += flood_fill(robo, visitadas, x,     y - 1)
    return conta


def celulas_alcancaveis(robo):
    visitadas = set()
    return flood_fill(robo, visitadas, robo['x'], robo['y'])


robo_livre = {'x': 0, 'y': 0, 'obstaculos': {}}
print(celulas_alcancaveis(robo_livre))   # 25 (grade 5x5 sem obstáculos)

robo_com_parede = {
    'x': 0, 'y': 0,
    'obstaculos': {(2, 0): True, (2, 1): True, (2, 2): True, (2, 3): True, (2, 4): True},
}
print(celulas_alcancaveis(robo_com_parede))   # 10 (parede vertical em x=2)


25
10


### Sua vez

In [7]:
# celulas_alcancaveis_de(robo, x0, y0) — flood fill a partir de um ponto qualquer.
def celulas_alcancaveis_de(robo, x0, y0):
    visitadas = set()
    return flood_fill(robo, visitadas, x0, y0)


print(celulas_alcancaveis_de(robo_com_parede, 4, 4))   # 10 (mesmo lado da parede)
print(celulas_alcancaveis_de(robo_com_parede, 0, 0))   # 10


10
10


## 4. `try/except`: capturar exceções específicas

`except:` nu captura **tudo**, inclusive bugs de programação (`NameError`, `TypeError`) —
esconde erros reais em vez de só tratar o que se espera. Regra: sempre nomeie a exceção.
Exceções diferentes pedem tratamentos diferentes — por isso vários `except` específicos, não
um genérico.


In [8]:
def parsear_seguro(linha):
    try:
        partes = linha.split()
        acao = partes[0]
        valor = int(partes[1])
        return acao, valor
    except ValueError:
        print(f"Valor não numérico na linha: {linha!r}")
        return None
    except IndexError:
        print(f"Linha mal formatada (esperado 'ACAO VALOR'): {linha!r}")
        return None


print(parsear_seguro("AVANCAR 3"))   # ('AVANCAR', 3)
print(parsear_seguro("GIRAR ESQ"))   # None (ValueError capturado)
print(parsear_seguro(""))            # None (IndexError capturado)


('AVANCAR', 3)
Valor não numérico na linha: 'GIRAR ESQ'
None
Linha mal formatada (esperado 'ACAO VALOR'): ''
None


### Sua vez

In [9]:
# dividir_seguro(a, b) — captura ZeroDivisionError.
def dividir_seguro(a, b):
    try:
        return a / b
    except ZeroDivisionError:
        print(f"Não é possível dividir {a} por zero.")
        return None


print(dividir_seguro(10, 2))   # 5.0
print(dividir_seguro(10, 0))   # None (com mensagem)


5.0
Não é possível dividir 10 por zero.
None


## 5. `FileNotFoundError` e exceções customizadas (teaser da D2)

`FileNotFoundError` aparece ao tentar abrir um arquivo que não existe. E às vezes as
exceções genéricas do Python (`ValueError`, `IndexError`) não bastam — não dizem nada sobre
o **domínio do robô**. `class NomeQualquer(Exception): pass` já é uma exceção válida, que se
comporta como qualquer outra: pode ser levantada com `raise` e capturada com `except`
(inclusive por categoria, se uma herda da outra). O que é uma `class` de verdade — e por que
isso funciona — vocês vão ver por completo na D2.


In [10]:
try:
    with open("arquivo_que_nao_existe.txt") as f:
        conteudo = f.read()
except FileNotFoundError:
    print("Arquivo de programa não encontrado!")


class RoboError(Exception):
    pass


class ParedeError(RoboError):
    pass


def avancar_ou_reclamar(robo, novo_x):
    if not (0 <= novo_x < LADO_GRADE):
        raise ParedeError(f"bateu na parede leste em x={novo_x}")
    robo['x'] = novo_x


robo_teste = {'x': 0}
try:
    avancar_ou_reclamar(robo_teste, 15)
except RoboError as erro:      # captura ParedeError — é subclasse de RoboError
    print(f"Erro do robô: {erro}")


Arquivo de programa não encontrado!
Erro do robô: bateu na parede leste em x=15


### Sua vez

In [11]:
# BateriaFracaError — levantada quando o nível está abaixo de 10.
class BateriaFracaError(Exception):
    pass


def verificar_bateria(nivel):
    if nivel < 10:
        raise BateriaFracaError(f"nível em {nivel}%, abaixo do mínimo seguro")


try:
    verificar_bateria(5)
except BateriaFracaError as erro:
    print(f"Bateria fraca: {erro}")


Bateria fraca: nível em 5%, abaixo do mínimo seguro


## 6. Arquivos texto: `with open` para ler e gravar

`with open(...)` garante que o arquivo é fechado automaticamente ao sair do bloco, mesmo se
houver erro — não precisa de `f.close()`. Para ler, iterar sobre o arquivo dá uma linha por
vez (use `.strip()`/`.rstrip()` para remover o `\n` do fim). Para gravar, `"w"` sobrescreve
o arquivo (cria se não existir); `"a"` acrescenta ao final.


In [12]:
# Ler linha a linha (log_aula03.txt está na mesma pasta deste notebook)
with open("log_aula03.txt") as arq:
    linhas = [linha.strip() for linha in arq if linha.strip()]

print(linhas)

# Gravar (sobrescreve se já existir)
trajetoria_exemplo = [(0, 0), (1, 0), (2, 0)]
with open("log_exemplo.txt", "w") as arq:
    for x, y in trajetoria_exemplo:
        arq.write(f"{x},{y}\n")

print("Log de exemplo gravado.")


['0,0', '0,3', '0,3', '2,3', '2,3', '2,4']
Log de exemplo gravado.


### Sua vez

In [13]:
# contar_posicoes_distintas(caminho) — usa um set para eliminar repetições.
def contar_posicoes_distintas(caminho):
    with open(caminho) as arq:
        posicoes = {linha.strip() for linha in arq if linha.strip()}
    return len(posicoes)


print(contar_posicoes_distintas("log_aula03.txt"))


4


## 7. Capstone: robô v6 lê o programa e grava o log

Três responsabilidades: `carregar_programa` lê e parseia o arquivo; `executar` processa os
comandos e acumula a trajetória; `gravar_log` persiste. Um detalhe: `parsear_seguro` sempre
devolve um inteiro como valor (passos para `AVANCAR`, +1/-1 de giro para `GIRAR`), mas
`avancar`/`girar` (seção de setup, herdadas do v4) não entendem inteiro direto — por isso dois
adaptadores finos (`avancar_n`, `girar_delta`) por cima delas, sem tocar no v4.


In [14]:
def avancar_n(robo, passos):
    for _ in range(passos):
        if not avancar(robo, robo['obstaculos']):
            break


def girar_delta(robo, delta):
    if delta == 1:
        girar(robo, "DIR")
    elif delta == -1:
        girar(robo, "ESQ")


def executar(robo, comandos):
    trajetoria = [(robo['x'], robo['y'])]
    tabela = {'AVANCAR': avancar_n, 'GIRAR': girar_delta}
    for acao, valor in comandos:
        if acao in tabela:
            tabela[acao](robo, valor)
            trajetoria.append((robo['x'], robo['y']))
    return trajetoria


def carregar_programa(caminho):
    comandos = []
    try:
        with open(caminho) as arq:
            for linha in arq:
                linha = linha.strip()
                if not linha or linha == "PARAR":
                    break
                cmd = parsear_seguro(linha)
                if cmd is not None:
                    comandos.append(cmd)
    except FileNotFoundError:
        print(f"Arquivo não encontrado: {caminho!r}")
    return comandos


def gravar_log(trajetoria, caminho="log.txt"):
    with open(caminho, "w") as log:
        for x, y in trajetoria:
            log.write(f"{x},{y}\n")
    print(f"Log gravado em '{caminho}' ({len(trajetoria)} posições).")


robo = {
    'x': 0, 'y': 0, 'direcao': 'NORTE', 'trajetoria': [(0, 0)],
    'obstaculos': {(2, 2): True},
}
programa = carregar_programa("../dados/aula-03/programa.txt")
trajetoria = executar(robo, programa)
gravar_log(trajetoria)
print(f"Robô terminou em ({robo['x']}, {robo['y']}), direção {robo['direcao']}.")
print(f"Células alcançáveis a partir desta posição: {celulas_alcancaveis(robo)}")


Log gravado em 'log.txt' (6 posições).
Robô terminou em (2, 4), direção NORTE.
Células alcançáveis a partir desta posição: 24


### Sua vez

In [15]:
# contar_comandos(caminho) — conta linhas antes do "PARAR", sem executar.
def contar_comandos(caminho):
    total = 0
    with open(caminho) as arq:
        for linha in arq:
            linha = linha.strip()
            if not linha or linha == "PARAR":
                break
            total += 1
    return total


print(contar_comandos("../dados/aula-03/programa.txt"))   # 5


5


## Para aprofundar

- **Recursão — introdução visual:** https://realpython.com/python-recursion/
- **Recursão — Python Tutor (visualizar a pilha):** https://pythontutor.com
- **Exceções — tutorial oficial:** https://docs.python.org/3/tutorial/errors.html
- **Arquivos — tutorial oficial:** https://docs.python.org/3/tutorial/inputoutput.html#reading-and-writing-files
- **`with` (gerenciador de contexto):** https://realpython.com/python-with-statement/
